# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`mbse/Patterns/Conformance/Corpus.py`, mirrored statement for statement
in TypeScript) and commits its snapshots under `conformance/<implementation>/`. This suite checks this implementation's
files are current, that its JSON is byte-identical to every other implementation's, and that it reads every other
implementation's JSON and YAML back to the same sets. Regenerate with `uv run python -m mbse.Patterns.Conformance.write`.

In [ ]:
import * as Y from "yaml";
import { readdirSync, readFileSync, statSync } from "node:fs";

import { CASES, build } from "../src/Conformance/Corpus.js";
import { render } from "../src/Conformance/write.js";
import { JSON as SchemaJSON, Plain, YAML } from "@mbse/schemas/Framework";
import { assert, same } from "./support.js";

const ROOT = "../../conformance";
const OWN = "typescript5";
const OTHERS = readdirSync(ROOT).filter((n) => n !== OWN && statSync(`${ROOT}/${n}`).isDirectory()).sort();
assert(same(OTHERS, ["python3"]), OTHERS.join());

const corpus = build();
const plain = (value: unknown) => JSON.stringify(value, (_, v) => (typeof v === "bigint" ? `${v}n` : v));

## CONF-01 · This implementation's committed files are current

In [ ]:
for (const [name, text] of render(corpus)) {
  assert(readFileSync(`${ROOT}/${OWN}/${name}`, "utf8") === text, `${name} is stale: regenerate the corpus`);
}

## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
{
  const expected = CASES.flatMap((c) => [`${c}.json`, `${c}.yaml`]).sort();
  for (const name of [OWN, ...OTHERS]) assert(same(readdirSync(`${ROOT}/${name}`).sort(), expected), name);
  for (const other of OTHERS) {
    for (const name of CASES) {
      const mine = readFileSync(`${ROOT}/${OWN}/${name}.json`);
      const theirs = readFileSync(`${ROOT}/${other}/${name}.json`);
      assert(mine.equals(theirs), `${name}.json differs from ${other}`);
    }
  }
}

## CONF-03 · This implementation reads every implementation's JSON and YAML back to the same sets

In [ ]:
for (const other of [OWN, ...OTHERS]) {
  for (const [name, [schema, root, store]] of corpus) {
    const expected = plain(Plain.ToPlain(store).Reachable(schema, root));
    const fromJson = SchemaJSON.FromJSON(store).Reachable(schema, readFileSync(`${ROOT}/${other}/${name}.json`, "utf8"));
    const fromYaml = YAML.FromYAML(store).Reachable(schema, readFileSync(`${ROOT}/${other}/${name}.yaml`, "utf8"));
    assert(plain(Plain.ToPlain(store).Reachable(schema, fromJson as any)) === expected, `${other} ${name} json`);
    assert(plain(Plain.ToPlain(store).Reachable(schema, fromYaml as any)) === expected, `${other} ${name} yaml`);
  }
}